# Clean Raw Hub

Reads from this notebook's `input/` folder and writes to its `output/`, `reports/`, and `dropped_and_dupes/` folders (all siblings of this notebook file). Each of the four sections below is self-contained (its own schema constants, `clean_xx()` function, and `df_raw_xx`/`df_cleaned_xx` variables) and can be re-run independently without clobbering another section's results.


## Imports & shared helpers

`find_default_input`, `month_tag_from_filename`, `write_report`, `strip_html`, and `read_semicolon_csv_protecting_backslashes` are identical in shape/logic across the four source notebooks. Here they're defined once, parameterized by `directory`/`prefix`/`filename_re` (and similar) instead of closing over notebook-global constants, and each of the four sections below calls these same functions with its own values.

Schema constants, `clean_xx()` logic, and rename/sort rules genuinely differ per dataset and are kept written out separately in each section rather than hidden behind a generic abstraction.


In [1]:
import csv
import html
import io
import re
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
REPORTS_DIR = NOTEBOOK_DIR / "reports"
DROPPED_DIR = NOTEBOOK_DIR / "dropped_and_dupes"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
DROPPED_DIR.mkdir(parents=True, exist_ok=True)


### find_default_input

Looks for a single `{prefix}_yyyy-mm-dd.{ext}` file in a given directory and returns it automatically. If none or several are found, it raises rather than silently guessing which one to use. `ext` defaults to `"csv"`; `HubAssessmentResults` (section 5) passes `ext="xlsx"` since its raw export is a genuine Excel file, not delimited text.


In [2]:
def find_default_input(directory: Path, prefix: str, filename_re: re.Pattern, ext: str = "csv") -> Path:
    matches = sorted(p for p in directory.glob(f"{prefix}_*.{ext}") if filename_re.match(p.name))
    if not matches:
        raise FileNotFoundError(f"No {prefix}_yyyy-mm-dd.{ext} file found in {directory}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple candidate input files found in {directory}: "
            f"{[m.name for m in matches]}. Pass one explicitly."
        )
    return matches[0]


### month_tag_from_filename

`month_tag` is `yyyymm` for the month *before* the input filename's `yyyy-mm-dd` date suffix (the export date's month minus one). Independent of any raw date data column. `ext` only affects the error message on a non-matching filename; the actual matching is driven entirely by `filename_re`.


In [3]:
def month_tag_from_filename(path: Path, prefix: str, filename_re: re.Pattern, ext: str = "csv") -> str:
    match = filename_re.match(path.name)
    if not match:
        raise ValueError(f"Filename '{path.name}' does not match expected pattern {prefix}_yyyy-mm-dd.{ext}")
    year, month = (int(g) for g in match.groups())
    # month_tag refers to the prior month's data, not the export date's month.
    year, month = (year - 1, 12) if month == 1 else (year, month - 1)
    return f"{year}{month:02d}"


### write_report

Writes the plain-text summary report: input filename, raw row/duplicate counts, `month_tag`, blank/missing-key rows dropped, duplicate rows collapsed, cleaned row/duplicate counts, output filename, then (after three blank lines) `df_cleaned.describe()`. `df_before_dedup` is the cleaned-but-not-yet-deduplicated frame (the output of `clean_xx()`), used so "blank/missing-key rows dropped" keeps its original meaning instead of conflating it with collapsed duplicates. Duplicate counts use pandas' default `duplicated()` (`keep="first"`) — the number of rows that would go away if the dataframe were deduplicated; `Cleaned duplicate rows` should always be `0` after the collapse step below, since every section's `GROUP_COLS` is "every `LS_COLS` field except the summed measures," making post-groupby rows unique by construction. Saved to `reports_dir` as `{prefix}_{month_tag}_report.txt`. Returns `(report_path, report_text)`.


In [4]:
def write_report(
    reports_dir: Path,
    prefix: str,
    month_tag: str,
    input_path: Path,
    df_raw: pd.DataFrame,
    df_before_dedup: pd.DataFrame,
    df_cleaned: pd.DataFrame,
    output_path: Path,
    before_sums: dict[str, int],
    after_sums: dict[str, int],
) -> tuple[Path, str]:
    report_lines = [
        f"Input file: {input_path.name}",
        f"Raw row count: {len(df_raw)}",
        f"Raw duplicate rows: {int(df_raw.duplicated().sum())}",
        "=======================================================================",
        f"Month tag: {month_tag}",
        f"Blank/missing-key rows dropped: {len(df_raw) - len(df_before_dedup)}",
        f"Duplicate rows collapsed: {len(df_before_dedup) - len(df_cleaned)}",
        "=======================================================================",
        f"Cleaned row count: {len(df_cleaned)}",
        f"Cleaned duplicate rows: {int(df_cleaned.duplicated().sum())}",
        f"Output file: {output_path.name}",
        "=======================================================================",
        "Numeric field sums, before dedup (post blank-drop) vs after dedup:",
    ]

    if before_sums:
        label_width = max(len(col) for col in before_sums)
        all_match = True
        for col, before_val in before_sums.items():
            after_val = after_sums[col]
            match = before_val == after_val
            all_match = all_match and match
            report_lines.append(
                f"  {col:<{label_width}}  before={before_val:<15} after={after_val:<15} "
                f"{'MATCH' if match else 'MISMATCH'}"
            )
        report_lines.append(f"All numeric sums match: {all_match}")
    else:
        report_lines.append("  (no numeric fields for this dataset)")

    report_lines.append("=======================================================================")

    report_text = "\n".join(report_lines) + "\n"
    report_text += "\n\n\n" + df_cleaned.describe().to_string() + "\n"

    reports_dir.mkdir(parents=True, exist_ok=True)
    report_path = reports_dir / f"{prefix}_{month_tag}_report.txt"
    report_path.write_text(report_text, encoding="utf-8")

    return report_path, report_text


### write_dropped_and_dupes

Writes every row dropped for being blank/missing-key, plus every row belonging to a collapsed duplicate group (all N rows per group, not just the N-1 that get summed away), into one plain (comma-delimited) CSV per dataset — tagged with a `DropReason` column (`"dropped blank"` or `"duplicate collapsed"`) so the two causes stay distinguishable. The two row sets can have different columns (blank rows keep their pre-clean raw columns; duplicate rows are already in `LS_COLS` shape), so they're combined with an outer-join concat rather than requiring identical schemas. Saved to `dropped_dir` as `{prefix}_{month_tag}_dropped_and_dupes.csv`.


In [5]:
def write_dropped_and_dupes(
    dropped_dir: Path,
    prefix: str,
    month_tag: str,
    dropped_blank: pd.DataFrame,
    duplicate_rows: pd.DataFrame,
) -> Path:
    dropped_blank = dropped_blank.copy()
    dropped_blank.insert(0, "DropReason", "dropped blank")

    duplicate_rows = duplicate_rows.copy()
    duplicate_rows.insert(0, "DropReason", "duplicate collapsed")

    combined = pd.concat([dropped_blank, duplicate_rows], ignore_index=True, sort=False)

    dropped_dir.mkdir(parents=True, exist_ok=True)
    out_path = dropped_dir / f"{prefix}_{month_tag}_dropped_and_dupes.csv"
    combined.to_csv(out_path, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

    return out_path


### strip_html

Generic HTML cleanup shared by the `HubDailyContent` and `HubDailyEvents` sections: strips tags (including ones with attributes, e.g. `<p style="...">`) via a `<...>` regex, unescapes entities (e.g. `&nbsp;`), and collapses the resulting non-breaking spaces to plain spaces.


In [6]:
HTML_TAG_RE = re.compile(r"<[^>]+>")


def strip_html(value: str) -> str:
    text = HTML_TAG_RE.sub("", value)
    text = html.unescape(text)
    return text.replace("\xa0", " ").strip()


def blank_out_null_text(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize literal "null"/"Null"/"NULL" text entries (a Hub export quirk, distinct
    from a genuinely empty cell) to blank, so downstream blank/missing-key checks and the
    cleaned output treat them the same as an empty cell.
    """
    df = df.copy()
    for col in df.columns:
        is_null_text = df[col].str.strip().str.lower() == "null"
        df.loc[is_null_text, col] = ""
    return df


def sum_int_cols(df: pd.DataFrame, cols: list[str]) -> dict[str, int]:
    """Sum each of `cols` (casting to int first, so this works whether or not the caller
    already cast the column) -- used to compare measure totals before/after dedup.
    """
    return {col: int(df[col].astype(int).sum()) for col in cols}

### read_semicolon_csv_protecting_backslashes

Shared by the `HubDailyContent` and `HubDailyEvents` sections, both of which need `engine="python", escapechar="\\"` to parse legitimate backslash-escaped quotes inside HTML attributes (e.g. `<p style=\"...\">`). Left unguarded, `escapechar` strips *every* backslash it precedes, not just ones before a quote — and both raw files contain a real company name, `TBWA\RAAD` (a genuine single backslash, not a CSV escape artifact), which would otherwise be corrupted to `TBWARAAD`. This pre-processes the raw text to double any backslash *not* immediately followed by `"`, so `escapechar` only ever consumes genuine `\"` sequences and every other backslash survives intact.


In [7]:
def read_semicolon_csv_protecting_backslashes(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # Protect literal backslashes that aren't a genuine CSV \" escape by doubling them,
    # so escapechar only ever consumes actual \" sequences below.
    protected_text = re.sub(r'\\(?!")', r"\\\\", raw_text)

    return pd.read_csv(
        io.StringIO(protected_text), sep=";", engine="python", escapechar="\\",
        dtype=str, keep_default_na=False, encoding="utf-8",
    )


## 1. HubDailyContent

Cleans a raw `HubDailyContentData_yyyy-mm-dd.csv` export. Input files use a `...Data` prefix, but output/report files use the shorter `HubDailyContent` prefix (per the notes' own worked example) — see `INPUT_PREFIX_DC` vs `PREFIX_DC` below.


### Schema constants

In [8]:
LS_COLS_DC = [
    "Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
    "AppInterest", "CategoryName", "ContentTitleLocalLanguage", "ContentTitle", "ContentLanguage",
    "ContentType", "DeviceCategory", "UserType", "Users", "TotalEvents", "UniqueEvents",
    "Stack", "Route", "Topic",
]
LS_STRING_COLS_DC = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
    "AppInterest", "CategoryName", "ContentTitleLocalLanguage", "ContentTitle", "ContentLanguage",
    "ContentType", "DeviceCategory", "UserType", "Users", "TotalEvents", "UniqueEvents",
    "Stack", "Route", "Topic",
]
LS_INT_COLS_DC = ["Users", "TotalEvents", "UniqueEvents"]
HTML_COLS_DC = ["ContentTitleLocalLanguage", "ContentTitle", "Stack"]

RENAME_MAP_DC = {"ContentTitle": "ContentTitleLocalLanguage", "ContentTitleEN": "ContentTitle"}

INPUT_PREFIX_DC = "HubDailyContentData"
PREFIX_DC = "HubDailyContent"
FILENAME_RE_DC = re.compile(r"^HubDailyContentData_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS_DC field except the summed
# measures (Users/TotalEvents/UniqueEvents) and summing those.
GROUP_COLS_DC = [c for c in LS_COLS_DC if c not in LS_INT_COLS_DC]
SUM_COLS_DC = LS_INT_COLS_DC


### Cleaning logic

1. Rename `ContentTitle` → `ContentTitleLocalLanguage` and `ContentTitleEN` → `ContentTitle` (raw local-language/English titles map onto `LS_COLS_DC`'s `ContentTitleLocalLanguage`/`ContentTitle`).
2. Drop rows that are blank across every `LS_COLS_DC` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Reorder/drop columns to match `LS_COLS_DC`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_DC` fields (this also includes `Users`/`TotalEvents`/`UniqueEvents`, ahead of the int cast below).
7. Cast `Users`, `TotalEvents`, `UniqueEvents` to integer type.
8. Strip HTML tags/attributes and unescape HTML entities on `ContentTitleLocalLanguage`, `ContentTitle`, `Stack`.


In [9]:
def clean_dc(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = blank_out_null_text(df)
    df = df.rename(columns=RENAME_MAP_DC)

    present_ls_cols = [c for c in LS_COLS_DC if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    dropped_blank = df.loc[is_blank].copy()
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    dropped_missing_key = df.loc[missing_key].copy()
    df = df.loc[~missing_key].copy()

    dropped = pd.concat([dropped_blank, dropped_missing_key])

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[[c for c in LS_COLS_DC if c in df.columns]]

    for col in LS_STRING_COLS_DC:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_DC:
        df[col] = df[col].astype(int)

    for col in HTML_COLS_DC:
        df[col] = df[col].apply(strip_html)

    return df, dropped


### Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS_DC` value are collapsed into one row, summing `Users`/`TotalEvents`/`UniqueEvents` as integers.


In [10]:
def collapse_duplicates_dc(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    duplicate_rows = df.loc[df.duplicated(subset=GROUP_COLS_DC, keep=False)].copy()

    for col in SUM_COLS_DC:
        df[col] = df[col].astype(int)
    df = df.groupby(GROUP_COLS_DC, as_index=False)[SUM_COLS_DC].sum()
    return df[LS_COLS_DC], duplicate_rows


### Configure the input file

Leave `INPUT_FILE_DC` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.


In [11]:
INPUT_FILE_DC = None  # e.g. "input/HubDailyContentData_2026-08-02.csv"

input_path_dc = Path(INPUT_FILE_DC).resolve() if INPUT_FILE_DC else find_default_input(INPUT_DIR, INPUT_PREFIX_DC, FILENAME_RE_DC)
month_tag_dc = month_tag_from_filename(input_path_dc, INPUT_PREFIX_DC, FILENAME_RE_DC)
input_path_dc, month_tag_dc


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubDailyContentData_2026-09-02.csv'),
 '202608')

### Read the raw CSV

Uses the shared `read_semicolon_csv_protecting_backslashes` helper (needs `escapechar` for HTML-attribute quotes, protected against corrupting `TBWA\RAAD`).


In [12]:
df_raw_dc = read_semicolon_csv_protecting_backslashes(input_path_dc)
df_raw_dc.shape


(9393, 30)

### Apply the cleaning steps

In [13]:
df_cleaned_dc, dropped_blank_dc = clean_dc(df_raw_dc)
df_cleaned_dc.head()


,Date,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,AppInterest,CategoryName,ContentTitleLocalLanguage,...,ContentLanguage,ContentType,DeviceCategory,UserType,Users,TotalEvents,UniqueEvents,Stack,Route,Topic
0,2026-08-07 00:00:00.000,ORANGEHUB,Orange Botswana,Botswana,Botswana,BW,ICAS Botswana,MentalHealth,Workplace,What are the symptoms of burnout?,...,en,Article,Mobile,Returning User,2,2,2,Beating burnout,Explore,Burnout
1,2026-08-23 00:00:00.000,ORANGEHUB,Orange Botswana,Botswana,Botswana,BW,ICAS Botswana,Lifestyle,,Emotional regulation tips for men and women,...,en,Article,Mobile,Returning User,1,1,1,What's new on Hub,Explore,Emotional agility/emotional intelligence
2,2026-08-07 00:00:00.000,ORANGEHUB,Orange Botswana,Botswana,Botswana,BW,ICAS Botswana,"Lifestyle,MentalHealth,Healthcare",,6 ways to create a better sleep environment,...,en,Article,Mobile,Returning User,1,1,1,Better sleep,Explore,Sleep and rest
3,2026-08-07 00:00:00.000,ORANGEHUB,Orange Botswana,Botswana,Botswana,BW,ICAS Botswana,Lifestyle,,Why is financial wellbeing important?,...,en,Article,Mobile,Returning User,1,1,1,Financial wellbeing,Explore,Money management
4,2026-08-07 00:00:00.000,ORANGEHUB,Orange Botswana,Botswana,Botswana,BW,ICAS Botswana,Lifestyle,,How we form our relationships with our finances,...,en,Article,Mobile,Returning User,2,2,2,Financial wellbeing,Explore,Money management


### Collapse duplicate rows before saving

`df_before_dedup_dc` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [14]:
df_before_dedup_dc = df_cleaned_dc
df_cleaned_dc, duplicate_rows_dc = collapse_duplicates_dc(df_before_dedup_dc)
duplicates_collapsed_dc = len(df_before_dedup_dc) - len(df_cleaned_dc)

print(f"Collapsed {duplicates_collapsed_dc} duplicate rows -> {len(df_cleaned_dc)} rows remaining")


Collapsed 122 duplicate rows -> 9269 rows remaining


### Save the cleaned dataset

In [15]:
output_path_dc = OUTPUT_DIR / f"{PREFIX_DC}_{month_tag_dc}_cleaned.csv"
df_cleaned_dc.to_csv(output_path_dc, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_dc)} rows -> {output_path_dc}")


Cleaned 9269 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubDailyContent_202608_cleaned.csv


### Write dropped/duplicate rows

In [16]:
dropped_path_dc = write_dropped_and_dupes(DROPPED_DIR, PREFIX_DC, month_tag_dc, dropped_blank_dc, duplicate_rows_dc)

print(f"Dropped/duplicate rows written -> {dropped_path_dc}")


Dropped/duplicate rows written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\dropped_and_dupes\HubDailyContent_202608_dropped_and_dupes.csv


### Write summary report

In [17]:
before_sums_dc = sum_int_cols(df_before_dedup_dc, SUM_COLS_DC)
after_sums_dc = sum_int_cols(df_cleaned_dc, SUM_COLS_DC)

report_path_dc, report_text_dc = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_DC,
    month_tag=month_tag_dc,
    input_path=input_path_dc,
    df_raw=df_raw_dc,
    df_before_dedup=df_before_dedup_dc,
    df_cleaned=df_cleaned_dc,
    output_path=output_path_dc,
    before_sums=before_sums_dc,
    after_sums=after_sums_dc,
)

print(report_text_dc)
print(f"Report written -> {report_path_dc}")


Input file: HubDailyContentData_2026-09-02.csv
Raw row count: 9393
Raw duplicate rows: 95
Month tag: 202608
Blank/missing-key rows dropped: 2
Duplicate rows collapsed: 122
Cleaned row count: 9269
Cleaned duplicate rows: 0
Output file: HubDailyContent_202608_cleaned.csv
Numeric field sums, before dedup (post blank-drop) vs after dedup:
  Users         before=9476            after=9476            MATCH
  TotalEvents   before=10169           after=10169           MATCH
  UniqueEvents  before=9588            after=9588            MATCH
All numeric sums match: True



             Users  TotalEvents  UniqueEvents
count  9269.000000  9269.000000   9269.000000
mean      1.022333     1.097098      1.034416
std       0.175168     0.394258      0.214902
min       1.000000     1.000000      1.000000
25%       1.000000     1.000000      1.000000
50%       1.000000     1.000000      1.000000
75%       1.000000     1.000000      1.000000
max       6.000000     8.000000      6.000000

Report written 

## 2. HubDailyEvents

Cleans a raw `HubDailyEventData_yyyy-mm-dd.csv` export. Like `HubDailyContent`, input files use a `...Data` prefix but output/report files don't — see `INPUT_PREFIX_DE` vs `PREFIX_DE` below.


### Schema constants

In [18]:
LS_COLS_DE = [
    "Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode",
    "Operation", "Feature", "EventAction", "EventCategory", "EventLabel", "EventSection",
    "EventQuestion", "UserType", "Users", "TotalEvents", "UniqueEvents",
    "SessionsWithEvent", "Events/SessionwithEvent",
]
LS_STRING_COLS_DE = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
    "EventAction", "EventCategory", "EventLabel", "EventSection", "EventQuestion", "UserType",
]
LS_INT_COLS_DE = ["TotalEvents", "UniqueEvents", "SessionsWithEvent", "Events/SessionwithEvent"]
HTML_COLS_DE = ["EventSection"]

RENAME_MAP_DE = {
    "eventAction": "EventAction",
    "eventCategory": "EventCategory",
    "eventLabel": "EventLabel",
    "eventSection": "EventSection",
    "eventQuestion": "EventQuestion",
    "EventsPerSessionWithEvent": "Events/SessionwithEvent",
}

SORT_COLS_DE = ["Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode"]

INPUT_PREFIX_DE = "HubDailyEventData"
PREFIX_DE = "HubDailyEvents"
FILENAME_RE_DE = re.compile(r"^HubDailyEventData_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS_DE field except the summed
# measures and summing those. Users isn't in LS_INT_COLS_DE (clean_de never casts it),
# so collapse_duplicates_de casts it defensively before summing.
SUM_COLS_DE = ["Users"] + LS_INT_COLS_DE
GROUP_COLS_DE = [c for c in LS_COLS_DE if c not in SUM_COLS_DE]


### Cleaning logic

1. Rename `eventAction`→`EventAction`, `eventCategory`→`EventCategory`, `eventLabel`→`EventLabel`, `eventSection`→`EventSection`, `eventQuestion`→`EventQuestion`, `EventsPerSessionWithEvent`→`Events/SessionwithEvent`.
2. Drop rows that are blank across every `LS_COLS_DE` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Strip HTML tags/attributes and unescape HTML entities on `EventSection` — before the whitespace pass below, so tag-removal artifacts get collapsed too.
6. Insert a new `Feature` column (always blank — no source data for it), positioned between `Operation` and `EventAction` per `LS_COLS_DE`.
7. Reorder/drop columns to match `LS_COLS_DE`.
8. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_DE` fields.
9. Cast `TotalEvents`, `UniqueEvents`, `SessionsWithEvent`, `Events/SessionwithEvent` to integer type.
10. Sort ascending by `SORT_COLS_DE`.


In [19]:
def clean_de(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = blank_out_null_text(df)
    df = df.rename(columns=RENAME_MAP_DE)

    present_ls_cols = [c for c in LS_COLS_DE if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    dropped_blank = df.loc[is_blank].copy()
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    dropped_missing_key = df.loc[missing_key].copy()
    df = df.loc[~missing_key].copy()

    dropped = pd.concat([dropped_blank, dropped_missing_key])

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    for col in HTML_COLS_DE:
        df[col] = df[col].apply(strip_html)

    df["Feature"] = ""

    df = df[LS_COLS_DE]

    for col in LS_STRING_COLS_DE:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_DE:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS_DE, ascending=True).reset_index(drop=True)

    return df, dropped


### Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS_DE` value are collapsed into one row, summing `SUM_COLS_DE` (`Users`, `TotalEvents`, `UniqueEvents`, `SessionsWithEvent`, `Events/SessionwithEvent`) as integers.


In [20]:
def collapse_duplicates_de(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    duplicate_rows = df.loc[df.duplicated(subset=GROUP_COLS_DE, keep=False)].copy()

    for col in SUM_COLS_DE:
        df[col] = df[col].astype(int)
    df = df.groupby(GROUP_COLS_DE, as_index=False)[SUM_COLS_DE].sum()
    return df[LS_COLS_DE], duplicate_rows


### Configure the input file

Leave `INPUT_FILE_DE` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.


In [21]:
INPUT_FILE_DE = None  # e.g. "input/HubDailyEventData_2026-08-02.csv"

input_path_de = Path(INPUT_FILE_DE).resolve() if INPUT_FILE_DE else find_default_input(INPUT_DIR, INPUT_PREFIX_DE, FILENAME_RE_DE)
month_tag_de = month_tag_from_filename(input_path_de, INPUT_PREFIX_DE, FILENAME_RE_DE)
input_path_de, month_tag_de


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubDailyEventData_2026-09-02.csv'),
 '202608')

### Read the raw CSV

Uses the shared `read_semicolon_csv_protecting_backslashes` helper (needs `escapechar` for the escaped quotes in `eventLabel`, protected against corrupting `TBWA\RAAD`).


In [22]:
df_raw_de = read_semicolon_csv_protecting_backslashes(input_path_de)
df_raw_de.shape


(71218, 26)

### Apply the cleaning steps

In [23]:
df_cleaned_de, dropped_blank_de = clean_de(df_raw_de)
df_cleaned_de.head()


,Date,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,Feature,EventAction,EventCategory,EventLabel,EventSection,EventQuestion,UserType,Users,TotalEvents,UniqueEvents,SessionsWithEvent,Events/SessionwithEvent
0,2026-08-01 00:00:00.000,,,Australia,,,,,link,,,,,Returning User,1,3,2,1,3
1,2026-08-01 00:00:00.000,,,Austria,,,,,link,,,,,Returning User,1,3,2,2,3
2,2026-08-01 00:00:00.000,,,Canada,,,,,link,,,,,Returning User,1,4,4,2,4
3,2026-08-01 00:00:00.000,,,Canada,,,,,button,,,,,Returning User,1,2,2,2,2
4,2026-08-01 00:00:00.000,,,France,,,,,link,,,,,Returning User,1,18,13,13,18


### Collapse duplicate rows before saving

`df_before_dedup_de` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [24]:
df_before_dedup_de = df_cleaned_de
df_cleaned_de, duplicate_rows_de = collapse_duplicates_de(df_before_dedup_de)
duplicates_collapsed_de = len(df_before_dedup_de) - len(df_cleaned_de)

print(f"Collapsed {duplicates_collapsed_de} duplicate rows -> {len(df_cleaned_de)} rows remaining")


Collapsed 1 duplicate rows -> 71215 rows remaining


### Save the cleaned dataset

In [25]:
output_path_de = OUTPUT_DIR / f"{PREFIX_DE}_{month_tag_de}_cleaned.csv"
df_cleaned_de.to_csv(output_path_de, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_de)} rows -> {output_path_de}")


Cleaned 71215 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubDailyEvents_202608_cleaned.csv


### Write dropped/duplicate rows

In [26]:
dropped_path_de = write_dropped_and_dupes(DROPPED_DIR, PREFIX_DE, month_tag_de, dropped_blank_de, duplicate_rows_de)

print(f"Dropped/duplicate rows written -> {dropped_path_de}")


Dropped/duplicate rows written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\dropped_and_dupes\HubDailyEvents_202608_dropped_and_dupes.csv


### Write summary report

In [27]:
before_sums_de = sum_int_cols(df_before_dedup_de, SUM_COLS_DE)
after_sums_de = sum_int_cols(df_cleaned_de, SUM_COLS_DE)

report_path_de, report_text_de = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_DE,
    month_tag=month_tag_de,
    input_path=input_path_de,
    df_raw=df_raw_de,
    df_before_dedup=df_before_dedup_de,
    df_cleaned=df_cleaned_de,
    output_path=output_path_de,
    before_sums=before_sums_de,
    after_sums=after_sums_de,
)

print(report_text_de)
print(f"Report written -> {report_path_de}")


Input file: HubDailyEventData_2026-09-02.csv
Raw row count: 71218
Raw duplicate rows: 0
Month tag: 202608
Blank/missing-key rows dropped: 2
Duplicate rows collapsed: 1
Cleaned row count: 71215
Cleaned duplicate rows: 0
Output file: HubDailyEvents_202608_cleaned.csv
Numeric field sums, before dedup (post blank-drop) vs after dedup:
  Users                    before=81536           after=81536           MATCH
  TotalEvents              before=133580          after=133580          MATCH
  UniqueEvents             before=100331          after=100331          MATCH
  SessionsWithEvent        before=93579           after=93579           MATCH
  Events/SessionwithEvent  before=133081          after=133081          MATCH
All numeric sums match: True



              Users   TotalEvents  UniqueEvents  SessionsWithEvent  Events/SessionwithEvent
count  71215.000000  71215.000000  71215.000000       71215.000000             71215.000000
mean       1.144927      1.875728      1.408846           1.3

## 3. HubDailyUsers

Cleans a raw `HubDailyUsers_yyyy-mm-dd.csv` export. Input and output prefixes are the same here, unlike `HubDailyContent`/`HubDailyEvents`.


### Schema constants

In [28]:
LS_COLS_DU = [
    "Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode",
    "Operation", "DeviceCategory", "UserType", "Users", "Sessions", "SessionDuration",
]
LS_STRING_COLS_DU = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode",
    "Operation", "DeviceCategory", "UserType",
]
LS_INT_COLS_DU = ["Sessions", "Users"]
SORT_COLS_DU = ["Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode"]

PREFIX_DU = "HubDailyUsers"
FILENAME_RE_DU = re.compile(r"^HubDailyUsers_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS_DU field except the summed
# measures and summing those. SessionDuration is a hh:mm:ss string, not numeric -- see
# the hms helpers in "Collapse duplicate rows" below.
SUM_COLS_DU = ["Users", "Sessions", "SessionDuration"]
GROUP_COLS_DU = [c for c in LS_COLS_DU if c not in SUM_COLS_DU]


### Cleaning logic

1. Drop the raw `SessionDuration` column and rename `SessionDurationInSeconds` → `SessionDuration` (despite its name, the raw `SessionDurationInSeconds` column actually holds `hh:mm:ss`-formatted strings, not a count of seconds — left as a string, not converted).
2. Drop rows that are blank across every `LS_COLS_DU` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Reorder/drop columns to match `LS_COLS_DU`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_DU` fields.
7. Cast `Sessions`, `Users` to integer type.
8. Sort ascending by `SORT_COLS_DU`.


In [29]:
def clean_du(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = blank_out_null_text(df)

    df = df.drop(columns=["SessionDuration"]).rename(columns={"SessionDurationInSeconds": "SessionDuration"})

    present_ls_cols = [c for c in LS_COLS_DU if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    dropped_blank = df.loc[is_blank].copy()
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    dropped_missing_key = df.loc[missing_key].copy()
    df = df.loc[~missing_key].copy()

    dropped = pd.concat([dropped_blank, dropped_missing_key])

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[[c for c in LS_COLS_DU if c in df.columns]]

    for col in LS_STRING_COLS_DU:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_DU:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS_DU, ascending=True).reset_index(drop=True)

    return df, dropped


### Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS_DU` value are collapsed into one row, summing `Users` and `Sessions` as integers. `SessionDuration` is a `hh:mm:ss` string, so it's summed by converting each value to total seconds, adding those, then formatting back to `hh:mm:ss` — hours are left unbounded (not wrapped at 24h) since this is an accumulated duration, not a clock time.


In [30]:
def _hms_to_seconds(value: str) -> int:
    h, m, s = value.split(":")
    return int(h) * 3600 + int(m) * 60 + int(s)


def _seconds_to_hms(total_seconds: int) -> str:
    h, remainder = divmod(total_seconds, 3600)
    m, s = divmod(remainder, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def collapse_duplicates_du(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    duplicate_rows = df.loc[df.duplicated(subset=GROUP_COLS_DU, keep=False)].copy()

    df["Users"] = df["Users"].astype(int)
    df["Sessions"] = df["Sessions"].astype(int)
    df = df.groupby(GROUP_COLS_DU, as_index=False).agg({
        "Users": "sum",
        "Sessions": "sum",
        "SessionDuration": lambda s: _seconds_to_hms(sum(_hms_to_seconds(v) for v in s)),
    })
    return df[LS_COLS_DU], duplicate_rows


### Configure the input file

Leave `INPUT_FILE_DU` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.


In [31]:
INPUT_FILE_DU = None  # e.g. "input/HubDailyUsers_2026-08-02.csv"

input_path_du = Path(INPUT_FILE_DU).resolve() if INPUT_FILE_DU else find_default_input(INPUT_DIR, PREFIX_DU, FILENAME_RE_DU)
month_tag_du = month_tag_from_filename(input_path_du, PREFIX_DU, FILENAME_RE_DU)
input_path_du, month_tag_du


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubDailyUsers_2026-09-02.csv'),
 '202608')

### Read the raw CSV

Plain read, no `engine`/`escapechar` — this file has no HTML content needing escaped quotes, and empirically those options only corrupt a real company name (`TBWA\RAAD` → `TBWARAAD`).


In [32]:
df_raw_du = pd.read_csv(input_path_du, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df_raw_du.shape


(39868, 21)

### Apply the cleaning steps

In [33]:
df_cleaned_du, dropped_blank_du = clean_du(df_raw_du)
df_cleaned_du.head()


,Date,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,DeviceCategory,UserType,Users,Sessions,SessionDuration
0,2026-08-01 00:00:00.000,,,Albania,,,,Mobile,Returning User,1,1,00:00:00
1,2026-08-01 00:00:00.000,,,Albania,,,,Desktop,Returning User,1,2,11:12:02
2,2026-08-01 00:00:00.000,,,Albania,,,,Mobile,Returning User,1,1,00:00:00
3,2026-08-01 00:00:00.000,,,Albania,,,,Mobile,Returning User,1,1,00:00:00
4,2026-08-01 00:00:00.000,,,Algeria,,,,Desktop,Returning User,1,1,00:00:00


### Collapse duplicate rows before saving

`df_before_dedup_du` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [34]:
df_before_dedup_du = df_cleaned_du
df_cleaned_du, duplicate_rows_du = collapse_duplicates_du(df_before_dedup_du)
duplicates_collapsed_du = len(df_before_dedup_du) - len(df_cleaned_du)

print(f"Collapsed {duplicates_collapsed_du} duplicate rows -> {len(df_cleaned_du)} rows remaining")


Collapsed 13362 duplicate rows -> 26504 rows remaining


### Save the cleaned dataset

In [35]:
output_path_du = OUTPUT_DIR / f"{PREFIX_DU}_{month_tag_du}_cleaned.csv"
df_cleaned_du.to_csv(output_path_du, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_du)} rows -> {output_path_du}")


Cleaned 26504 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubDailyUsers_202608_cleaned.csv


### Write dropped/duplicate rows

In [36]:
dropped_path_du = write_dropped_and_dupes(DROPPED_DIR, PREFIX_DU, month_tag_du, dropped_blank_du, duplicate_rows_du)

print(f"Dropped/duplicate rows written -> {dropped_path_du}")


Dropped/duplicate rows written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\dropped_and_dupes\HubDailyUsers_202608_dropped_and_dupes.csv


### Write summary report

In [37]:
# SessionDuration is a hh:mm:ss string, not an int column -- sum it separately in
# seconds via the same helpers collapse_duplicates_du uses, rather than through
# sum_int_cols (which assumes every column is directly int-castable).
before_sums_du = sum_int_cols(df_before_dedup_du, ["Users", "Sessions"])
before_sums_du["SessionDuration (sec)"] = sum(_hms_to_seconds(v) for v in df_before_dedup_du["SessionDuration"])
after_sums_du = sum_int_cols(df_cleaned_du, ["Users", "Sessions"])
after_sums_du["SessionDuration (sec)"] = sum(_hms_to_seconds(v) for v in df_cleaned_du["SessionDuration"])

report_path_du, report_text_du = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_DU,
    month_tag=month_tag_du,
    input_path=input_path_du,
    df_raw=df_raw_du,
    df_before_dedup=df_before_dedup_du,
    df_cleaned=df_cleaned_du,
    output_path=output_path_du,
    before_sums=before_sums_du,
    after_sums=after_sums_du,
)

print(report_text_du)
print(f"Report written -> {report_path_du}")


Input file: HubDailyUsers_2026-09-02.csv
Raw row count: 39868
Raw duplicate rows: 1509
Month tag: 202608
Blank/missing-key rows dropped: 2
Duplicate rows collapsed: 13362
Cleaned row count: 26504
Cleaned duplicate rows: 0
Output file: HubDailyUsers_202608_cleaned.csv
Numeric field sums, before dedup (post blank-drop) vs after dedup:
  Users                  before=70648           after=70648           MATCH
  Sessions               before=145494          after=145494          MATCH
  SessionDuration (sec)  before=453405396       after=453405396       MATCH
All numeric sums match: True



              Users      Sessions
count  26504.000000  26504.000000
mean       2.665560      5.489511
std       11.113773     29.819016
min        1.000000      1.000000
25%        1.000000      1.000000
50%        1.000000      1.000000
75%        2.000000      3.000000
max      435.000000   1932.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\reports\Hub

## 4. HubMonthlyUsers

Cleans a raw `HubMonthlyUsers_yyyy-mm-dd.csv` export. The structural odd one out: no `escapechar` (same `TBWA\RAAD` reasoning as `HubDailyUsers`), no HTML columns, no sort step, and an extra `Month` column reformat that the other three sections don't have.


### Schema constants

In [38]:
LS_COLS_MU = [
    "MonthDate", "Month", "CompanyCode", "CompanyName", "Country",
    "HomeCountry", "HomeCountryCode", "Operation",
    "NewUsers", "Users", "Sessions", "Hits",
]
LS_STRING_COLS_MU = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
]
LS_INT_COLS_MU = ["NewUsers", "Users", "Sessions", "Hits"]

PREFIX_MU = "HubMonthlyUsers"
FILENAME_RE_MU = re.compile(r"^HubMonthlyUsers_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS_MU field except the summed
# measures (NewUsers/Users/Sessions/Hits) and summing those.
GROUP_COLS_MU = [c for c in LS_COLS_MU if c not in LS_INT_COLS_MU]
SUM_COLS_MU = LS_INT_COLS_MU


### Cleaning logic

1. Rename `Date` → `MonthDate`.
2. Drop rows that are blank across every `LS_COLS_MU` field present in the raw data.
3. Drop rows where `MonthDate` is blank.
4. Reformat `MonthDate` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds) and `Month` to `%Y %m`.
5. Reorder/drop columns to match `LS_COLS_MU`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_MU` fields.
7. Cast `NewUsers`, `Users`, `Sessions`, `Hits` to integer type.

No sort step here, unlike the other three sections — preserved faithfully, not "fixed" to match.


In [39]:
def clean_mu(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = blank_out_null_text(df)
    df = df.rename(columns={"Date": "MonthDate"})

    present_ls_cols = [c for c in LS_COLS_MU if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    dropped_blank = df.loc[is_blank].copy()
    df = df.loc[~is_blank].copy()

    missing_key = df["MonthDate"].str.strip() == ""
    dropped_missing_key = df.loc[missing_key].copy()
    df = df.loc[~missing_key].copy()

    dropped = pd.concat([dropped_blank, dropped_missing_key])

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["MonthDate"] = pd.to_datetime(df["MonthDate"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]
    df["Month"] = pd.to_datetime(df["Month"], format="%Y-%m").dt.strftime("%Y %m")

    df = df[[c for c in LS_COLS_MU if c in df.columns]]

    for col in LS_STRING_COLS_MU:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_MU:
        df[col] = df[col].astype(int)

    return df, dropped


### Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS_MU` value are collapsed into one row, summing `NewUsers`/`Users`/`Sessions`/`Hits` as integers.


In [40]:
def collapse_duplicates_mu(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    duplicate_rows = df.loc[df.duplicated(subset=GROUP_COLS_MU, keep=False)].copy()

    for col in SUM_COLS_MU:
        df[col] = df[col].astype(int)
    df = df.groupby(GROUP_COLS_MU, as_index=False)[SUM_COLS_MU].sum()
    return df[LS_COLS_MU], duplicate_rows


### Configure the input file

Leave `INPUT_FILE_MU` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.


In [41]:
INPUT_FILE_MU = None  # e.g. "input/HubMonthlyUsers_2026-08-02.csv"

input_path_mu = Path(INPUT_FILE_MU).resolve() if INPUT_FILE_MU else find_default_input(INPUT_DIR, PREFIX_MU, FILENAME_RE_MU)
month_tag_mu = month_tag_from_filename(input_path_mu, PREFIX_MU, FILENAME_RE_MU)
input_path_mu, month_tag_mu


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubMonthlyUsers_2026-09-02.csv'),
 '202608')

### Read the raw CSV

Plain read, no `engine`/`escapechar` — same reasoning as `HubDailyUsers`: this file has no HTML content, and those options would corrupt the real company name `TBWA\RAAD`.


In [42]:
df_raw_mu = pd.read_csv(input_path_mu, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df_raw_mu.shape


(5980, 20)

### Apply the cleaning steps

In [43]:
df_cleaned_mu, dropped_blank_mu = clean_mu(df_raw_mu)
df_cleaned_mu.head()


,MonthDate,Month,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,NewUsers,Users,Sessions,Hits
0,2026-08-01 00:00:00.000,2026 08,UNILEVER,Unilever,Costa Rica,Colombia,CO,ICAS Latina,0,1,1,3
1,2026-08-01 00:00:00.000,2026 08,GEVERNOVA,GE Vernova,Mexico,Colombia,CO,ICAS Latina,0,1,2,38
2,2026-08-01 00:00:00.000,2026 08,CVANGUARDIA,CERRO VANGUARDIA,Argentina,Argentina,AR,ICAS Latina,2,3,3,69
3,2026-08-01 00:00:00.000,2026 08,CUMMINS,Cummins,Argentina,Argentina,AR,ICAS Latina,4,5,5,23
4,2026-08-01 00:00:00.000,2026 08,CUMMINS,Cummins,United States of America,Argentina,AR,ICAS Latina,2,2,2,2


### Collapse duplicate rows before saving

`df_before_dedup_mu` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [44]:
df_before_dedup_mu = df_cleaned_mu
df_cleaned_mu, duplicate_rows_mu = collapse_duplicates_mu(df_before_dedup_mu)
duplicates_collapsed_mu = len(df_before_dedup_mu) - len(df_cleaned_mu)

print(f"Collapsed {duplicates_collapsed_mu} duplicate rows -> {len(df_cleaned_mu)} rows remaining")


Collapsed 136 duplicate rows -> 5842 rows remaining


### Save the cleaned dataset

In [45]:
output_path_mu = OUTPUT_DIR / f"{PREFIX_MU}_{month_tag_mu}_cleaned.csv"
df_cleaned_mu.to_csv(output_path_mu, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_mu)} rows -> {output_path_mu}")


Cleaned 5842 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubMonthlyUsers_202608_cleaned.csv


### Write dropped/duplicate rows

In [46]:
dropped_path_mu = write_dropped_and_dupes(DROPPED_DIR, PREFIX_MU, month_tag_mu, dropped_blank_mu, duplicate_rows_mu)

print(f"Dropped/duplicate rows written -> {dropped_path_mu}")


Dropped/duplicate rows written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\dropped_and_dupes\HubMonthlyUsers_202608_dropped_and_dupes.csv


### Write summary report

In [47]:
before_sums_mu = sum_int_cols(df_before_dedup_mu, SUM_COLS_MU)
after_sums_mu = sum_int_cols(df_cleaned_mu, SUM_COLS_MU)

report_path_mu, report_text_mu = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_MU,
    month_tag=month_tag_mu,
    input_path=input_path_mu,
    df_raw=df_raw_mu,
    df_before_dedup=df_before_dedup_mu,
    df_cleaned=df_cleaned_mu,
    output_path=output_path_mu,
    before_sums=before_sums_mu,
    after_sums=after_sums_mu,
)

print(report_text_mu)
print(f"Report written -> {report_path_mu}")


Input file: HubMonthlyUsers_2026-09-02.csv
Raw row count: 5980
Raw duplicate rows: 0
Month tag: 202608
Blank/missing-key rows dropped: 2
Duplicate rows collapsed: 136
Cleaned row count: 5842
Cleaned duplicate rows: 0
Output file: HubMonthlyUsers_202608_cleaned.csv
Numeric field sums, before dedup (post blank-drop) vs after dedup:
  NewUsers  before=43027           after=43027           MATCH
  Users     before=51068           after=51068           MATCH
  Sessions  before=145426          after=145426          MATCH
  Hits      before=367391          after=367391          MATCH
All numeric sums match: True



          NewUsers        Users      Sessions          Hits
count  5842.000000  5842.000000   5842.000000   5842.000000
mean      7.365115     8.741527     24.893187     62.887881
std     104.593383   120.857475    361.690902    665.727034
min       0.000000     1.000000      1.000000      1.000000
25%       1.000000     1.000000      1.000000      2.000000
50%       1.000000     1

## 5. HubAssessmentResults

Cleans a raw `HubAssessmentResults_yyyy-mm-dd.xlsx` export (Sheet1) — the structural odd one out: the only section reading a genuine Excel file rather than a CSV (`pd.read_excel`, via the shared `find_default_input`/`month_tag_from_filename` helpers' `ext="xlsx"` parameter), no `escapechar`/backslash-protection needed since Excel cells aren't delimiter-parsed at all (there's no escape sequence to protect, and empirically zero literal backslashes exist in the data anyway), and no duplicate-collapsing step (per the notes) — `write_report` below is called with `df_cleaned_ar` for both dedup-related parameters, so `Duplicate rows collapsed` reports as `0` while the raw/cleaned duplicate counts still surface any that exist.


### Schema constants

In [48]:
LS_COLS_AR = [
    "Date", "CompanyCode", "CompanyName", "CurrentCountry", "HomeCountry",
    "Operation", "AssessmentType", "SectionName", "Result",
]
LS_STRING_COLS_AR = [
    "CompanyCode", "CompanyName", "CurrentCountry", "HomeCountry",
    "Operation", "AssessmentType", "SectionName", "Result",
]
LS_INT_COLS_AR = []
HTML_COLS_AR = ["SectionName"]
SORT_COLS_AR = ["Date", "CompanyCode", "CompanyName", "CurrentCountry", "HomeCountry", "Operation"]

PREFIX_AR = "HubAssessmentResults"
FILENAME_RE_AR = re.compile(r"^HubAssessmentResults_(\d{4})-(\d{2})-\d{2}\.xlsx$")


### Cleaning logic

1. Drop rows that are blank across every `LS_COLS_AR` field present in the raw data.
2. Drop rows where `Date` is blank.
3. Reformat `Date` from its raw `yyyy-mm-dd hh:mm:ss.fffffff +00:00` shape to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds) — this dataset's raw timestamp already carries full time-of-day precision plus a UTC offset, unlike the plain `yyyy-mm-dd` dates the other four sections parse.
4. Strip HTML tags/entities from `SectionName` (shared `strip_html`).
5. Lowercase `Result`.
6. Reorder/drop columns to match `LS_COLS_AR`.
7. Trim and collapse whitespace on the `LS_STRING_COLS_AR` fields.
8. Cast `LS_INT_COLS_AR` to integer type (no-op — this dataset has none).
9. Sort ascending by `SORT_COLS_AR`.


In [49]:
def clean_ar(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = blank_out_null_text(df)

    present_ls_cols = [c for c in LS_COLS_AR if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    dropped_blank = df.loc[is_blank].copy()
    df = df.loc[~is_blank].copy()

    missing_key = df["Date"].str.strip() == ""
    dropped_missing_key = df.loc[missing_key].copy()
    df = df.loc[~missing_key].copy()

    dropped = pd.concat([dropped_blank, dropped_missing_key])

    # The raw timestamp carries 7-digit fractional seconds and a UTC offset (always
    # "+00:00"); %f zero-pads/truncates to 6-digit microseconds, and slicing off the
    # last 3 leaves milliseconds. Dropping the offset from the output format loses no
    # information since every row is already UTC.
    df["Date"] = (
        pd.to_datetime(df["Date"], format="%Y-%m-%d %H:%M:%S.%f %z")
        .dt.strftime("%Y-%m-%d %H:%M:%S.%f")
        .str[:-3]
    )

    for col in HTML_COLS_AR:
        df[col] = df[col].apply(strip_html)

    df["Result"] = df["Result"].str.lower()

    df = df[[c for c in LS_COLS_AR if c in df.columns]]

    for col in LS_STRING_COLS_AR:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_AR:
        df[col] = df[col].astype(int)

    df = df.sort_values(by=SORT_COLS_AR, ascending=True).reset_index(drop=True)

    return df, dropped


### Configure the input file

Leave `INPUT_FILE_AR` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.


In [50]:
INPUT_FILE_AR = None  # e.g. "input/HubAssessmentResults_2026-08-02.xlsx"

input_path_ar = Path(INPUT_FILE_AR).resolve() if INPUT_FILE_AR else find_default_input(INPUT_DIR, PREFIX_AR, FILENAME_RE_AR, ext="xlsx")
month_tag_ar = month_tag_from_filename(input_path_ar, PREFIX_AR, FILENAME_RE_AR, ext="xlsx")
input_path_ar, month_tag_ar


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubAssessmentResults_2026-09-02.xlsx'),
 '202608')

### Read the raw Excel file

Read everything as strings (`dtype=str`, `keep_default_na=False`) from `Sheet1`, so blank fields and literal `"NULL"` text pass through unchanged instead of being coerced or turned into `NaN`. No `read_semicolon_csv_protecting_backslashes`/`escapechar` handling here — `pd.read_excel` reads cell values directly rather than tokenizing a delimited text stream, so there's no escape sequence to protect in the first place.


In [51]:
df_raw_ar = pd.read_excel(input_path_ar, sheet_name="Sheet1", dtype=str, keep_default_na=False)
df_raw_ar.shape


(9237, 9)

### Apply the cleaning steps

In [52]:
df_cleaned_ar, dropped_blank_ar = clean_ar(df_raw_ar)
df_cleaned_ar.head()


,Date,CompanyCode,CompanyName,CurrentCountry,HomeCountry,Operation,AssessmentType,SectionName,Result
0,2026-08-01 00:43:47.955,ESK002,Eskom,South Africa,South Africa,Lyra Southern Africa Pty Ltd,checkIn,,sad
1,2026-08-01 01:30:39.830,SAP,SAP,United Kingdom,United Kingdom,Lyra Health International Ltd,checkIn,,devastated
2,2026-08-01 02:06:08.756,ARIES,Grupo Aries,Mexico,Mexico,Lyra Mexico and Central America,checkIn,,happy
3,2026-08-01 02:09:06.097,PEPSICO,Pepsico,New Zealand,New Zealand,Lyra Health International Ltd,checkIn,,happy
4,2026-08-01 02:33:09.193,PEPSICO,Pepsico,New Zealand,New Zealand,Lyra Health International Ltd,checkIn,,devastated


### Save the cleaned dataset

In [53]:
output_path_ar = OUTPUT_DIR / f"{PREFIX_AR}_{month_tag_ar}_cleaned.csv"
df_cleaned_ar.to_csv(output_path_ar, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_ar)} rows -> {output_path_ar}")


Cleaned 9237 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubAssessmentResults_202608_cleaned.csv


### Write dropped/duplicate rows

No dedup step for this dataset (per the notes), so there are no duplicate-collapsed rows to report — an empty frame is passed through for that side.


In [54]:
dropped_path_ar = write_dropped_and_dupes(DROPPED_DIR, PREFIX_AR, month_tag_ar, dropped_blank_ar, pd.DataFrame())

print(f"Dropped/duplicate rows written -> {dropped_path_ar}")


Dropped/duplicate rows written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\dropped_and_dupes\HubAssessmentResults_202608_dropped_and_dupes.csv


### Write summary report

In [55]:
# LS_INT_COLS_AR is empty (no numeric fields in this schema), so both sum dicts are empty.
report_path_ar, report_text_ar = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_AR,
    month_tag=month_tag_ar,
    input_path=input_path_ar,
    df_raw=df_raw_ar,
    df_before_dedup=df_cleaned_ar,
    df_cleaned=df_cleaned_ar,
    output_path=output_path_ar,
    before_sums={},
    after_sums={},
)

print(report_text_ar)
print(f"Report written -> {report_path_ar}")


Input file: HubAssessmentResults_2026-09-02.xlsx
Raw row count: 9237
Raw duplicate rows: 0
Month tag: 202608
Blank/missing-key rows dropped: 0
Duplicate rows collapsed: 0
Cleaned row count: 9237
Cleaned duplicate rows: 0
Output file: HubAssessmentResults_202608_cleaned.csv
Numeric field sums, before dedup (post blank-drop) vs after dedup:
  (no numeric fields for this dataset)



                           Date  CompanyCode                     CompanyName CurrentCountry   HomeCountry                      Operation AssessmentType SectionName   Result
count                      9237         9237                            9237           9237          9237                           9237           9237        9237     9237
unique                     9237          847                             845            112           120                             24              2           7       10
top     2026-08-01 00:43:47.955  V3STAGETEST  Velocity Cubed Staging Testing   South Africa  South

## Summary

Convenience recap of everything produced by this run.


In [56]:
print("Cleaned outputs:")
for label, out_path, rpt_path, drp_path in [
    ("DailyContent", output_path_dc, report_path_dc, dropped_path_dc),
    ("DailyEvents",  output_path_de, report_path_de, dropped_path_de),
    ("DailyUsers",   output_path_du, report_path_du, dropped_path_du),
    ("MonthlyUsers", output_path_mu, report_path_mu, dropped_path_mu),
    ("AssessmentResults", output_path_ar, report_path_ar, dropped_path_ar),
]:
    print(f"  {label:14s} -> {out_path.name}  (report: {rpt_path.name}, dropped/dupes: {drp_path.name})")


Cleaned outputs:
  DailyContent   -> HubDailyContent_202608_cleaned.csv  (report: HubDailyContent_202608_report.txt, dropped/dupes: HubDailyContent_202608_dropped_and_dupes.csv)
  DailyEvents    -> HubDailyEvents_202608_cleaned.csv  (report: HubDailyEvents_202608_report.txt, dropped/dupes: HubDailyEvents_202608_dropped_and_dupes.csv)
  DailyUsers     -> HubDailyUsers_202608_cleaned.csv  (report: HubDailyUsers_202608_report.txt, dropped/dupes: HubDailyUsers_202608_dropped_and_dupes.csv)
  MonthlyUsers   -> HubMonthlyUsers_202608_cleaned.csv  (report: HubMonthlyUsers_202608_report.txt, dropped/dupes: HubMonthlyUsers_202608_dropped_and_dupes.csv)
  AssessmentResults -> HubAssessmentResults_202608_cleaned.csv  (report: HubAssessmentResults_202608_report.txt, dropped/dupes: HubAssessmentResults_202608_dropped_and_dupes.csv)
